In [3]:
"""
Исследование Билибина через OpenRouter (GPT-4o) с сохранением прогресса.
При прерывании можно продолжить с того же места.
"""

import os, json, base64, warnings, time, colorsys
from io import BytesIO
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from openai import OpenAI
from sklearn.cluster import KMeans
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. КОНФИГУРАЦИЯ
# ------------------------------------------------------------
try:
    from my_config import openrouter_api
except ImportError:
    raise RuntimeError("Добавьте в my_config.py переменную openrouter_api = 'sk-or-v1-...'")

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

CONFIG = {
    "api_base_url": "https://openrouter.ai/api/v1",
    "model_name": "openai/gpt-4o",
    "api_key": openrouter_api,
    "group_1_path": BASE_DIR / "early_images",
    "group_2_path": BASE_DIR / "late_images",
    "output_file": BASE_DIR / "bilibin_analysis.csv",
    "max_images_early": 40,
    "max_images_late": 28,
    "palette_colors": 5,
    "image_max_size": 768,
    "image_quality": 70,
    "request_timeout": 60,
    "max_retries": 2,
    "save_every": 1,                # сохранять после каждого изображения
    "delay_between_requests": 0.5,  # небольшая пауза
    "resume": True,                 # продолжить с места остановки
}

# ------------------------------------------------------------
# 2. АЛГОРИТМИЧЕСКИЙ АНАЛИЗ
# ------------------------------------------------------------
class AlgorithmicAnalyzer:
    @staticmethod
    def get_dominant_colors(image_path, n_colors=5):
        img = Image.open(image_path).convert("RGB")
        img.thumbnail((200, 200))
        pixels = np.array(img).reshape(-1, 3)
        kmeans = KMeans(n_clusters=n_colors, random_state=42, n_init=10).fit(pixels)
        centers = kmeans.cluster_centers_.astype(int)
        labels = kmeans.labels_
        total_pixels = len(labels)
        percentages = [round(np.sum(labels == i) / total_pixels * 100, 1) for i in range(n_colors)]
        sorted_idx = np.argsort(percentages)[::-1]
        palette = []
        for idx in sorted_idx:
            r, g, b = centers[idx]
            palette.append({
                "hex": f"#{r:02x}{g:02x}{b:02x}",
                "rgb": [int(r), int(g), int(b)],
                "percentage": percentages[idx]
            })
        return palette

    @staticmethod
    def get_tonality(image_path):
        img = Image.open(image_path).convert("L")
        arr = np.array(img, dtype=np.float32)
        mean_brightness = arr.mean() / 255.0
        return round((mean_brightness - 0.5) * 2, 3)

    @staticmethod
    def get_color_harmony(image_path, n_colors=5):
        img = Image.open(image_path).convert("RGB")
        img.thumbnail((150, 150))
        pixels = np.array(img).reshape(-1, 3)
        kmeans = KMeans(n_clusters=n_colors, random_state=42, n_init=10).fit(pixels)
        centers = kmeans.cluster_centers_
        hues = []
        for r, g, b in centers:
            r, g, b = r/255., g/255., b/255.
            h, s, v = colorsys.rgb_to_hsv(r, g, b)
            hues.append(h)
        hues_sorted = sorted(hues)
        diffs = [(hues_sorted[i+1] - hues_sorted[i]) % 1.0 for i in range(len(hues_sorted)-1)]
        avg_diff = np.mean(diffs) if diffs else 0
        if avg_diff < 0.1: return "монохромная"
        elif 0.25 < avg_diff < 0.35: return "триадная"
        elif 0.4 < avg_diff < 0.6: return "комплементарная"
        return "аналогичная"

# ------------------------------------------------------------
# 3. LLM АНАЛИЗАТОР (GPT-4o)
# ------------------------------------------------------------
class LLMAnalyzer:
    def __init__(self, client, model_name):
        self.client = client
        self.model = model_name
        self._api_available = True

    def _check_connection(self):
        try:
            self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": "OK"}],
                max_tokens=2,
                timeout=CONFIG["request_timeout"]
            )
            return True
        except Exception as e:
            print(f"❌ Ошибка подключения: {e}")
            return False

    def encode_image_to_base64(self, image_path):
        with Image.open(image_path) as img:
            if img.mode != "RGB":
                img = img.convert("RGB")
            img.thumbnail((CONFIG["image_max_size"], CONFIG["image_max_size"]), Image.Resampling.LANCZOS)
            buf = BytesIO()
            img.save(buf, format="JPEG", quality=CONFIG["image_quality"])
            return base64.b64encode(buf.getvalue()).decode("utf-8")

    def analyze_semantic(self, image_path):
        if not self._api_available:
            return self._empty_result()

        b64 = self.encode_image_to_base64(image_path)

        prompt = (
            "Ты эксперт по живописи. Верни ТОЛЬКО JSON:\n"
            '{"genre":"","objects":[],"affect":"","dynamic_score":0,"perspective_type":"","depth_score":0,"composition_complexity":0,"graphic_vs_painterly":0}\n'
            "genre: жанр (историческая иллюстрация, пейзаж, портрет, сказочная сцена и т.д.)\n"
            "objects: список объектов\n"
            "affect: настроение (торжественное, зловещее, спокойное, радостное, таинственное)\n"
            "dynamic_score: от -1 (статика) до +1 (динамика)\n"
            "perspective_type: тип (линейная, воздушная, обратная, отсутствует, орнаментальная)\n"
            "depth_score: от 0 (плоскость) до 1 (глубокая сцена)\n"
            "composition_complexity: от 0 (простая) до 1 (очень сложная)\n"
            "graphic_vs_painterly: от -1 (мягкая живопись) до +1 (чёткая графика)"
        )

        messages = [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}}
            ]
        }]

        for attempt in range(CONFIG["max_retries"] + 1):
            try:
                resp = self.client.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    max_tokens=300,
                    temperature=0.0,
                    timeout=CONFIG["request_timeout"]
                )
                answer = resp.choices[0].message.content.strip()
                if answer.startswith("```"):
                    answer = answer.split("```")[1]
                    if answer.startswith("json"):
                        answer = answer[4:]
                data = json.loads(answer)
                for field in ["dynamic_score", "depth_score", "composition_complexity", "graphic_vs_painterly"]:
                    try: data[field] = float(data.get(field, 0))
                    except: data[field] = 0.0
                if not isinstance(data.get("objects"), list):
                    data["objects"] = []
                return data
            except Exception as e:
                print(f"⚠️ LLM ошибка (попытка {attempt+1}): {e}")
                if attempt < CONFIG["max_retries"]:
                    time.sleep(2)
                else:
                    return self._empty_result()
        return self._empty_result()

    def _empty_result(self):
        return {
            "genre": "", "objects": [], "affect": "",
            "dynamic_score": 0.0, "perspective_type": "", "depth_score": 0.0,
            "composition_complexity": 0.0, "graphic_vs_painterly": 0.0
        }

# ------------------------------------------------------------
# 4. УТИЛИТЫ СОХРАНЕНИЯ И ПРОГРЕССА
# ------------------------------------------------------------
def load_existing_results(output_path):
    """Загружает существующий CSV и возвращает DataFrame + множество обработанных файлов."""
    if not output_path.exists():
        return pd.DataFrame(), set()
    try:
        df = pd.read_csv(output_path, encoding="utf-8-sig")
        if "filename" not in df.columns:
            print("⚠️ В существующем файле нет колонки 'filename', возобновление невозможно.")
            return pd.DataFrame(), set()
        processed = set(df["filename"].dropna().unique())
        print(f"📂 Найдено существующих записей: {len(df)}, обработано файлов: {len(processed)}")
        return df, processed
    except Exception as e:
        print(f"⚠️ Ошибка чтения существующего файла: {e}")
        return pd.DataFrame(), set()

def save_dataframe(df, output_path):
    """Сохраняет DataFrame в CSV, создавая резервную копию предыдущего файла."""
    # Создаём бэкап, если файл существует
    if output_path.exists():
        backup_path = output_path.with_suffix(".csv.bak")
        try:
            output_path.replace(backup_path)
        except Exception:
            pass
    df.to_csv(output_path, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 5. КОНВЕЙЕР
# ------------------------------------------------------------
def process_image(llm, algo, filepath):
    palette = algo.get_dominant_colors(filepath, CONFIG["palette_colors"])
    tonality = algo.get_tonality(filepath)
    harmony = algo.get_color_harmony(filepath)
    sem = llm.analyze_semantic(filepath)
    return {
        "palette": json.dumps(palette, ensure_ascii=False),
        "tonality": tonality,
        "color_harmony": harmony,
        "genre": sem.get("genre", ""),
        "objects": json.dumps(sem.get("objects", []), ensure_ascii=False),
        "affect": sem.get("affect", ""),
        "dynamic_score": sem.get("dynamic_score", 0.0),
        "perspective_type": sem.get("perspective_type", ""),
        "depth_score": sem.get("depth_score", 0.0),
        "composition_complexity": sem.get("composition_complexity", 0.0),
        "graphic_vs_painterly": sem.get("graphic_vs_painterly", 0.0),
    }

def run_pipeline():
    print("Инициализация OpenRouter (GPT-4o)...")
    client = OpenAI(base_url=CONFIG["api_base_url"], api_key=CONFIG["api_key"])
    llm = LLMAnalyzer(client, CONFIG["model_name"])
    algo = AlgorithmicAnalyzer()

    print("Проверка подключения...")
    if not llm._check_connection():
        print("❌ API недоступен. LLM-поля будут пустыми.")
        llm._api_available = False
    else:
        print("✅ API доступен.")

    output_path = Path(CONFIG["output_file"])
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Возобновление прогресса
    processed_files = set()
    results_df = pd.DataFrame()
    if CONFIG["resume"]:
        results_df, processed_files = load_existing_results(output_path)

    # Формируем список групп для обработки
    groups = [
        (CONFIG["group_1_path"], "early", CONFIG["max_images_early"]),
        (CONFIG["group_2_path"], "late", CONFIG["max_images_late"]),
    ]

    total_processed = len(results_df)
    for folder, group_label, limit in groups:
        folder = Path(folder)
        if not folder.is_dir():
            print(f"Папка {folder} не найдена, пропускаем {group_label}")
            continue

        all_files = sorted([f for f in folder.iterdir() if f.suffix.lower() in ('.png', '.jpg', '.jpeg')])
        # Исключаем уже обработанные
        to_process = [f for f in all_files[:limit] if f.name not in processed_files]

        if not to_process:
            print(f"Группа '{group_label}': все файлы уже обработаны.")
            continue

        print(f"\nГруппа '{group_label}': {len(to_process)} новых изображений из {len(all_files[:limit])}")

        for filepath in tqdm(to_process, desc=f"Анализ {group_label}"):
            # Небольшая пауза
            time.sleep(CONFIG["delay_between_requests"])

            try:
                features = process_image(llm, algo, filepath)
                row = {"group": group_label, "filename": filepath.name, **features}
                # Добавляем строку к DataFrame
                results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
                processed_files.add(filepath.name)
                total_processed += 1

                # Сохраняем после каждого изображения
                save_dataframe(results_df, output_path)

            except Exception as e:
                print(f"❌ Критическая ошибка на {filepath.name}: {e}")
                # Сохраняем хотя бы алгоритмические данные
                try:
                    palette = algo.get_dominant_colors(filepath, CONFIG["palette_colors"])
                    tonality = algo.get_tonality(filepath)
                    harmony = algo.get_color_harmony(filepath)
                    row = {
                        "group": group_label, "filename": filepath.name,
                        "palette": json.dumps(palette, ensure_ascii=False),
                        "tonality": tonality, "color_harmony": harmony,
                        "genre": "", "objects": "[]", "affect": "",
                        "dynamic_score": 0.0, "perspective_type": "", "depth_score": 0.0,
                        "composition_complexity": 0.0, "graphic_vs_painterly": 0.0,
                    }
                    results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
                    processed_files.add(filepath.name)
                    total_processed += 1
                    save_dataframe(results_df, output_path)
                except Exception as inner_e:
                    print(f"   Не удалось получить даже алгоритмические данные: {inner_e}")

    print(f"\n✅ Готово! Всего обработано строк: {total_processed}")
    print(f"Результаты сохранены в {output_path}")

if __name__ == "__main__":
    run_pipeline()

Инициализация OpenRouter (GPT-4o)...
Проверка подключения...
✅ API доступен.
📂 Найдено существующих записей: 2, обработано файлов: 2

Группа 'early': 39 новых изображений из 40


Анализ early: 100%|██████████| 39/39 [02:45<00:00,  4.23s/it]



Группа 'late': 27 новых изображений из 28


Анализ late:  85%|████████▌ | 23/27 [01:27<00:14,  3.59s/it]

⚠️ LLM ошибка (попытка 1): Error code: 402 - {'error': {'message': 'Prompt tokens limit exceeded: 1862 > 1467. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_3EPr2ie2mAjbxF2YcxMPTf0FL8n'}
⚠️ LLM ошибка (попытка 2): Error code: 402 - {'error': {'message': 'Prompt tokens limit exceeded: 1862 > 1467. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_3EPr2ie2mAjbxF2YcxMPTf0FL8n'}


...

⚠️ LLM ошибка (попытка 3): Error code: 402 - {'error': {'message': 'Prompt tokens limit exceeded: 1862 > 1467. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_3EPr2ie2mAjbxF2YcxMPTf0FL8n'}
⚠️ LLM ошибка (попытка 1): Error code: 402 - {'error': {'message': 'Prompt tokens limit exceeded: 1862 > 1467. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_3EPr2ie2mAjbxF2YcxMPTf0FL8n'}
⚠️ LLM ошибка (попытка 2): Error code: 402 - {'error': {'message': 'Prompt tokens limit exceeded: 1862 > 1467. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_3EPr2ie2mAjbxF2YcxMPTf0FL8n'}


Анализ late:  96%|█████████▋| 26/27 [01:47<00:05,  5.49s/it]

⚠️ LLM ошибка (попытка 3): Error code: 402 - {'error': {'message': 'Prompt tokens limit exceeded: 1862 > 1467. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_3EPr2ie2mAjbxF2YcxMPTf0FL8n'}
⚠️ LLM ошибка (попытка 1): Error code: 402 - {'error': {'message': 'Prompt tokens limit exceeded: 1862 > 1467. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_3EPr2ie2mAjbxF2YcxMPTf0FL8n'}
⚠️ LLM ошибка (попытка 2): Error code: 402 - {'error': {'message': 'Prompt tokens limit exceeded: 1862 > 1467. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_3EPr2ie2mAjbxF2YcxMPTf0FL8n'}


Анализ late: 100%|██████████| 27/27 [01:53<00:00,  4.20s/it]

⚠️ LLM ошибка (попытка 3): Error code: 402 - {'error': {'message': 'Prompt tokens limit exceeded: 1862 > 1467. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_3EPr2ie2mAjbxF2YcxMPTf0FL8n'}

✅ Готово! Всего обработано строк: 68
Результаты сохранены в C:\Users\kek20\PycharmProjects\research\bilibin_analysis.csv
